In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

# Check if TensorFlow was built with CUDA
print("Built with CUDA:", tf.test.is_built_with_cuda())

# List available GPUs
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", gpus)


# Stable Diffusion (PyTorch) – End-to-End Pipeline

This notebook:
- Uses **PyTorch only**
- Loads **Stable Diffusion v1.5**
- Generates images from text prompts
- Evaluates outputs using practical diffusion metrics
- Saves the **entire pipeline + weights**
- Allows reuse of the saved model in a **VS Code frontend (`main.py`)**

Frameworks:
- diffusers
- transformers
- torch
- safetensors


In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision safetensors ftfy pillow


## Import Libraries and Configure Device

We detect the best available device:
- CUDA (GPU)
- Apple MPS (Mac)
- CPU fallback


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import numpy as np
import os


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


## Load Stable Diffusion v1.5 (PyTorch)

- Loads pretrained weights from Hugging Face
- Uses Euler Ancestral scheduler for better quality
- Enables memory optimizations


In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,   # disable NSFW checker (optional)
)

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config
)

pipe = pipe.to(device)
pipe.enable_attention_slicing()

print("Pipeline loaded successfully")


## Text Prompt Definition

- `prompt` describes what we want
- `negative_prompt` tells the model what to avoid


In [ ]:
prompt = "ultra-detailed portrait of a red fox wearing a tiny scarf, cinematic lighting, 35mm"
negative_prompt = "blurry, lowres, jpeg artifacts, extra fingers, text, watermark"


## Prompt Enhancement (User → Professional Prompt)

Users usually give **short, vague prompts**.
Diffusion models perform best with:
- Subject clarity
- Style descriptors
- Camera & lighting
- Quality boosters

This cell:
- Takes a raw user prompt
- Enhances it into a **high-quality, professional diffusion prompt**
- Keeps it deterministic and controllable


In [ ]:
def enhance_prompt(
    user_prompt: str,
    style: str = "photorealistic",
    mood: str = "cinematic",
    camera: str = "35mm lens, shallow depth of field",
    lighting: str = "soft cinematic lighting, global illumination",
    quality: str = "ultra-detailed, 8k, high sharpness, professional color grading"
):
    """
    Enhances a short user prompt into a professional Stable Diffusion prompt.
    """

    enhanced_prompt = f"""
    {user_prompt},
    {style},
    {mood},
    {lighting},
    {camera},
    {quality},
    highly detailed textures,
    realistic anatomy,
    award-winning photography,
    masterpiece
    """

    # Clean formatting
    enhanced_prompt = " ".join(enhanced_prompt.split())
    return enhanced_prompt



## Example: Enhance User Prompt



In [ ]:
# Raw prompt from user (UI / frontend / input box)
user_prompt = "a red fox wearing a red scarf"

# Enhance it
enhanced_prompt = enhance_prompt(
    user_prompt,
    style="hyper-realistic animal portrait",
    mood="cinematic fantasy",
    camera="85mm DSLR lens, f1.8",
    lighting="dramatic rim lighting, soft shadows"
)

negative_prompt = (
    "blurry, lowres, jpeg artifacts, extra limbs, "
    "bad anatomy, text, watermark, logo"
)

print("USER PROMPT:\n", user_prompt)
print("\nENHANCED PROMPT:\n", enhanced_prompt)


# Prompt Engineering and Prompt Ensembling Techniques in CLIP
Implementattion of CLIP - https://github.com/AyushRanjan13/CLIP_Implementation

## Prompt Engineering

CLIP performs classification by comparing an image embedding with text embeddings generated from class descriptions. Instead of using raw class names such as:

```text
dog
cat
car
```

the CLIP paper shows that converting class labels into natural language prompts significantly improves performance. This process is known as **Prompt Engineering**.

For example, for the class **dog**, different prompts can be:

```text
dog
a photo of a dog.
a high quality photo of a dog.
a blurry photo of a dog.
an image of a dog.
```

Each prompt produces a different text embedding. Since CLIP was trained on image-caption pairs collected from the internet, natural language descriptions better match the data distribution seen during training. As a result, prompts such as **"a photo of a dog"** generally achieve higher zero-shot accuracy than using only the class name **"dog"**.

### Advantages of Prompt Engineering

* Improves zero-shot classification accuracy.
* Requires no additional training or fine-tuning.
* Leverages CLIP's understanding of natural language.
* Helps align text descriptions with real-world image captions.
* Simple and computationally inexpensive to implement.

---

## Prompt Ensembling

While a single prompt may work well, different prompts capture different semantic aspects of the same class. To make predictions more robust, the CLIP paper introduces **Prompt Ensembling**.

Instead of using only one prompt:

```text
a photo of a dog.
```

multiple prompts are created:

```text
a photo of a dog.
a picture of a dog.
a photograph of a dog.
a bright photo of a dog.
a photo of a small dog.
a photo of the dog.
```

The text embedding for each prompt is computed using CLIP's text encoder. These embeddings are then averaged in the embedding space and normalized to obtain a single representative embedding for the class.

For the class **dog**:

```text
Embedding(dog) =
Average[
    Embedding("a photo of a dog."),
    Embedding("a picture of a dog."),
    Embedding("a photograph of a dog."),
    ...
]
```

This averaged embedding is used during zero-shot classification.

### Why Prompt Ensembling Works

Different prompts emphasize different characteristics of an object. Some prompts may perform better on certain images, while others may perform better on different viewing conditions. Averaging multiple text representations reduces the bias of any single prompt and produces a more stable class representation.

### Advantages of Prompt Ensembling

* Produces more robust text embeddings.
* Reduces sensitivity to prompt wording.
* Improves zero-shot classification performance.
* Captures multiple semantic views of the same class.
* No additional training is required.
* The ensemble is computed once and cached, so inference cost remains nearly the same as using a single prompt.



In [ ]:
# ============================================================
# Prompt Engineering (Paper Section 3.1.4)
# ============================================================

# Different prompt templates — from the paper's discussion
PROMPT_TEMPLATES = {
    "Bare class name\n(Li et al. 2017 baseline)": "{label}",
    "Default photo prompt\n(+1.3% on ImageNet)": "a photo of a {label}.",
    "Context-rich prompt": "a photo of a {label}, a type of object.",
    "High-quality prompt": "a high quality photo of a {label}.",
    "Blurry photo prompt": "a blurry photo of a {label}.",
    "Art prompt": "an art drawing of a {label}.",
    "Low res prompt": "a low resolution photo of a {label}.",
}

print("Prompt Engineering Experiment (Paper Section 3.1.4)")
print("="*65)

prompt_results = {}
for template_name, template in PROMPT_TEMPLATES.items():
    # Encode class names with this template
    text_feats = encode_text_prompts(CIFAR10_CLASSES, template)
    preds, labels, _ = zero_shot_predict(test_loader, text_feats)
    acc = accuracy_score(labels, preds) * 100
    prompt_results[template_name] = acc
    print(f"  {template_name.split(chr(10))[0]:<35}: {acc:.2f}%")

# ── Visualise Results ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
names   = [n.replace('\n', ' ') for n in prompt_results.keys()]
accs    = list(prompt_results.values())
colors  = ['#e74c3c' if i == 0 else '#2ecc71' if i == 1 else '#3498db'
            for i in range(len(accs))]

bars = ax.bar(range(len(accs)), accs, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xticks(range(len(accs)))
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Zero-Shot Accuracy (%)', fontsize=11)
ax.set_title('Impact of Prompt Engineering on Zero-Shot CIFAR-10 Accuracy\n(Paper Section 3.1.4)',
              fontsize=12, fontweight='bold')
ax.set_ylim(min(accs) - 5, max(accs) + 5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='Bare class name (paper baseline)'),
    mpatches.Patch(color='#2ecc71', label='Default photo prompt (paper recommended)'),
    mpatches.Patch(color='#3498db', label='Custom prompts'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.axhline(y=accs[0], color='#e74c3c', linestyle='--', alpha=0.5, label='Baseline')

plt.tight_layout()
plt.show()

best_template = max(prompt_results, key=prompt_results.get)
print(f"\n Best Template : '{best_template.split(chr(10))[0]}'")
print(f"   Accuracy       : {prompt_results[best_template]:.2f}%")
print(f"   Gain over bare : +{prompt_results[best_template] - accs[0]:.2f}%")

In [ ]:
# ============================================================
# Prompt Ensembling (Paper Section 3.1.4)
# ============================================================

# Multiple prompt templates for ensembling (inspired by paper's 80-template ensemble)
ENSEMBLE_TEMPLATES = [
    "a photo of a {label}.",
    "a photograph of a {label}.",
    "a picture of a {label}.",
    "an image of a {label}.",
    "a photo of the {label}.",
    "a high quality photo of a {label}.",
    "a low resolution photo of a {label}.",
    "a good photo of a {label}.",
    "a photo of a big {label}.",      # From paper: 'A photo of a big {label}'
    "a photo of a small {label}.",    # From paper: 'A photo of a small {label}'
    "a photo of a nice {label}.",
    "a photo of a cool {label}.",
    "a jpeg corrupted photo of a {label}.",
    "a bright photo of a {label}.",
    "a dark photo of a {label}.",
    "a photo of my {label}.",
]


def build_ensemble_features(class_names, templates):
    """
    Build ensembled text features by averaging embeddings across templates.

    Paper (Section 3.1.4):
    "We construct the ensemble over the embedding space instead of
    probability space. This allows us to cache a single set of averaged
    text embeddings so that the compute cost of the ensemble is the
    same as using a single classifier."

    Returns:
        ensemble_features : Averaged + L2-normalised text embeddings [N_classes, D]
    """
    all_class_embeddings = []

    with torch.no_grad():
        for cls in class_names:
            # Embed this class under each template
            cls_prompts  = [t.format(label=cls) for t in templates]
            tokens       = clip.tokenize(cls_prompts).to(device)
            cls_features = model.encode_text(tokens)            # [N_templates, D]
            cls_features = F.normalize(cls_features, dim=-1)

            # Average across templates → single embedding per class
            cls_mean = cls_features.mean(dim=0)                 # [D]
            cls_mean = F.normalize(cls_mean, dim=-1)            # re-normalise
            all_class_embeddings.append(cls_mean)

    return torch.stack(all_class_embeddings)  # [N_classes, D]


# ── Compare: Single Prompt vs Ensemble ────────────────────────
print("Prompt Ensembling Experiment (Paper Section 3.1.4)")
print("="*60)

# Single prompt (best from previous cell)
single_feats  = encode_text_prompts(CIFAR10_CLASSES, "a photo of a {label}.")
preds_single, labels_s, _ = zero_shot_predict(test_loader, single_feats)
acc_single = accuracy_score(labels_s, preds_single) * 100

# Ensembled prompts
print(f"\n Building ensemble from {len(ENSEMBLE_TEMPLATES)} templates...")
ensemble_feats = build_ensemble_features(CIFAR10_CLASSES, ENSEMBLE_TEMPLATES)
preds_ens, labels_e, _ = zero_shot_predict(test_loader, ensemble_feats)
acc_ensemble = accuracy_score(labels_e, preds_ens) * 100

print(f"\n{'='*60}")
print(f"  Single prompt accuracy  : {acc_single:.2f}%")
print(f"  Ensemble accuracy       : {acc_ensemble:.2f}%")
print(f"  Gain from ensembling    : +{acc_ensemble - acc_single:.2f}%")
print(f"  (Paper reports +3.5% gain on ImageNet with 80 templates)")
print(f"{'='*60}")

# ── Visualise ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: single vs ensemble
axes[0].bar(['Single\nPrompt', f'Ensemble\n({len(ENSEMBLE_TEMPLATES)} prompts)'],
             [acc_single, acc_ensemble],
             color=['#3498db', '#2ecc71'], edgecolor='black')
axes[0].set_ylabel('Accuracy (%)', fontsize=11)
axes[0].set_title('Single vs Ensembled Prompts\n(Paper Section 3.1.4)', fontsize=11, fontweight='bold')
axes[0].set_ylim(acc_single - 5, max(acc_single, acc_ensemble) + 5)
for i, acc in enumerate([acc_single, acc_ensemble]):
    axes[0].text(i, acc + 0.3, f'{acc:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Confusion matrix for ensemble
cm = confusion_matrix(labels_e, preds_ens)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
axes[1].set_xlabel('Predicted', fontsize=10)
axes[1].set_ylabel('True', fontsize=10)
axes[1].set_title('Confusion Matrix\n(Ensemble Zero-Shot)', fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Image Generation

Important parameters:
- `num_inference_steps`: image quality vs speed
- `guidance_scale`: prompt adherence
- `seed`: reproducibility


In [ ]:
generator = torch.Generator(device=device).manual_seed(42)

image = pipe(
    prompt=enhanced_prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    height=512,
    width=512,
    generator=generator
).images[0]

image.save("generated_image_enhanced.png")
image


## Evaluation Metrics for Diffusion Models

Diffusion models don't use accuracy.
Common evaluation approaches:
1. **CLIP similarity score** (text–image alignment)
2. **Inference latency**
3. **Determinism (seed reproducibility)**

Below we compute CLIP similarity using OpenAI CLIP.


In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

inputs = clip_processor(
    text=[prompt],
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    clip_score = outputs.logits_per_image.item()

print("CLIP similarity score:", clip_score)


## Save Model and Weights (For VS Code / Frontend)

This saves:
- UNet
- VAE
- Text Encoder
- Scheduler
- Tokenizer

You can zip and download this folder.


In [ ]:
SAVE_DIR = "/content/stable_diffusion_saved"

pipe.save_pretrained(SAVE_DIR)
print(f"Model saved to: {SAVE_DIR}")


In [ ]:
import os
print(os.listdir("/content"))


In [ ]:

!mv /content/stable_diffusion_saved.zip /content/drive/MyDrive/
